In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

train = pd.read_csv("train_en.csv")
test = pd.read_csv("test_en.csv")
sub_sample = pd.read_csv("sample_submission_en.csv")

train.shape, test.shape

((542, 14), (163, 13))

In [2]:
train.head()

,id,gender,age,country,education_level,social_media_platform,avg_daily_usage_hours,academic_impact,bedtime,wake_time,mental_health_score,relationship_status,social_media_conflicts,addiction
0,ID_80,P,21,Maldives,S2,Instagram,4.2,No,20:02,03:50,8,berpasangan,2,No
1,ID_52,P,23,Bahamas,S2,LinkedIn,2.8,No,23:25,06:25,8,berpasangan,1,No
2,ID_486,female,21,Denmark,S1,Twitter,4.6,No,20:32,03:50,7,berpasangan,2,No
3,ID_203,perempuan,19,India,S1,WhatsApp,7.2,Yes,20:00,01:36,5,berpasangan,4,Yes
4,ID_317,laki-laki,21,Denmark,S2,Facebook,2.8,No,20:02,05:02,8,berpasangan,2,No


In [3]:
pred = ((train["mental_health_score"] <= 6) & (train["social_media_platform"] != "KakaoTalk"))
pred = pred.map({True: "Yes", False: "No"})

f1_score(train["addiction"], pred, average="macro")

0.9981061863414804

In [4]:
wrong = train[pred != train["addiction"]]
print(len(wrong))
wrong

1


,id,gender,age,country,education_level,social_media_platform,avg_daily_usage_hours,academic_impact,bedtime,wake_time,mental_health_score,relationship_status,social_media_conflicts,addiction
487,ID_126,male,23,Canada,S2,Instagram,5.2,Yes,23:28,06:28,6,berpasangan,3,No


In [5]:
cell = train[(train.mental_health_score == 6) & (train.social_media_conflicts == 3) & (train.academic_impact == "Yes")]
cell = cell[cell.social_media_platform != "KakaoTalk"]
print(len(cell))
cell["addiction"].value_counts()

144


,count
addiction,
Yes,143
No,1


In [6]:
print(pd.crosstab(cell.age, cell.addiction))
print()
print(pd.crosstab(cell.relationship_status, cell.addiction))
print()
print(pd.crosstab(cell.education_level, cell.addiction))
print()
print(pd.crosstab(cell.gender, cell.addiction))

addiction  No  Yes
age               
19          0   25
20          0   32
21          0   45
22          0   34
23          1    2
24          0    5

addiction            No  Yes
relationship_status         
berpasangan           1   31
hubungan rumit        0   10
lajang                0  102

addiction        No  Yes
education_level         
S1                0   61
S2                1   82

addiction  No  Yes
gender            
L           0   35
P           0   11
female      0   22
laki-laki   0   26
male        1   31
perempuan   0   18


In [7]:
def to_hours(t):
    h, m = t.split(":")
    return int(h) + int(m)/60

sleep = ((cell.wake_time.apply(to_hours) - cell.bedtime.apply(to_hours)) % 24)

cell.assign(sleep_hours=sleep).groupby("addiction")[["avg_daily_usage_hours"]].describe()

avg_daily_usage_hours                                             
                          count      mean       std  min  25%  50%  75%  max
addiction                                                                   
No                          1.0  5.200000       NaN  5.2  5.2  5.2  5.2  5.2
Yes                       143.0  5.183217  0.758333  3.5  4.7  5.2  5.7  7.3

In [8]:
cell.assign(sleep_hours=sleep).groupby("addiction")["sleep_hours"].describe()

,count,mean,std,min,25%,50%,75%,max
addiction,,,,,,,,
No,1.0,7.000000,NaN,7.0,7.0,7.0,7.0,7.0
Yes,143.0,6.590909,0.869508,4.6,5.9,6.6,7.0,8.6


In [9]:
print(train[train.social_media_platform == "KakaoTalk"].country.unique())
print(train[train.country == "South Korea"][["social_media_platform","mental_health_score","addiction"]])

['South Korea']
    social_media_platform  mental_health_score addiction
84              KakaoTalk                    6        No
102             KakaoTalk                    6        No
110             KakaoTalk                    6        No
309             Instagram                    7        No
314             KakaoTalk                    6        No
325             KakaoTalk                    6        No
357             KakaoTalk                    6        No
400             KakaoTalk                    6        No
425             KakaoTalk                    6        No
527             KakaoTalk                    6        No
537             KakaoTalk                    6        No


In [10]:
test[(test.mental_health_score == 6) & (test.social_media_conflicts == 3) & (test.academic_impact == "Yes")].social_media_platform.value_counts()

,count
social_media_platform,
TikTok,15
Instagram,12
WhatsApp,6
WeChat,2
KakaoTalk,2
Twitter,2
Facebook,1
Snapchat,1


In [11]:
def make_features(df):
    gender_map = {"P":"female","female":"female","perempuan":"female",
                  "L":"male","male":"male","laki-laki":"male"}
    f = pd.DataFrame(index=df.index)
    f["gender"] = df.gender.map(gender_map)
    f["age"] = df.age
    f["edu"] = df.education_level.map({"SMA":0,"S1":1,"S2":2})
    f["platform"] = df.social_media_platform
    f["is_kakao"] = (df.social_media_platform == "KakaoTalk").astype(int)
    f["usage"] = df.avg_daily_usage_hours
    f["academic"] = (df.academic_impact == "Yes").astype(int)
    f["sleep"] = ((df.wake_time.apply(to_hours) - df.bedtime.apply(to_hours)) % 24)
    f["mh"] = df.mental_health_score
    f["relationship"] = df.relationship_status
    f["conflicts"] = df.social_media_conflicts
    return f

Xtr_raw = make_features(train)
Xte_raw = make_features(test)
y = (train.addiction == "Yes").astype(int)

cats = ["gender","platform","relationship"]
nums = ["age","edu","usage","academic","sleep","mh","conflicts","is_kakao"]

Xtr = pd.get_dummies(Xtr_raw[cats+nums], columns=cats)
Xte = pd.get_dummies(Xte_raw[cats+nums], columns=cats).reindex(columns=Xtr.columns, fill_value=0)
Xtr.shape

(542, 25)

In [12]:
def rule(f):
    return ((f.mh <= 6) & (f.is_kakao == 0)).astype(int)

strat_key = y.astype(str) + "_" + Xtr_raw.is_kakao.astype(str)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

oof = {name: np.zeros(len(train)) for name in ["rule","logreg","tree","forest","lgbm"]}
n_seen = np.zeros(len(train))

for tr_i, va_i in cv.split(Xtr, strat_key):
    n_seen[va_i] += 1
    oof["rule"][va_i] += rule(Xtr_raw.iloc[va_i])

    sc = StandardScaler()
    Xs_tr, Xs_va = sc.fit_transform(Xtr.iloc[tr_i]), sc.transform(Xtr.iloc[va_i])
    lr = LogisticRegression(max_iter=1000).fit(Xs_tr, y.iloc[tr_i])
    oof["logreg"][va_i] += lr.predict(Xs_va)

    dt = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xtr.iloc[tr_i], y.iloc[tr_i])
    oof["tree"][va_i] += dt.predict(Xtr.iloc[va_i])

    rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr.iloc[tr_i], y.iloc[tr_i])
    oof["forest"][va_i] += rf.predict(Xtr.iloc[va_i])

    gbm = lgb.LGBMClassifier(n_estimators=200, max_depth=4, random_state=42, verbosity=-1).fit(Xtr.iloc[tr_i], y.iloc[tr_i])
    oof["lgbm"][va_i] += gbm.predict(Xtr.iloc[va_i])

{name: f1_score(y, (arr/n_seen >= 0.5).astype(int), average="macro") for name, arr in oof.items()}

{'rule': 0.9981061863414804,
 'logreg': 0.9981061863414804,
 'tree': 0.9981061863414804,
 'forest': 0.9981061863414804,
 'lgbm': 0.9962100552408922}

In [13]:
big_countries = train.country.value_counts()
big_countries = big_countries[big_countries >= 5].index

out = []
for c in big_countries:
    mask = (train.country == c).values
    dt = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xtr[~mask], y[~mask])
    score = f1_score(y[mask], dt.predict(Xtr[mask]), average="macro")
    out.append((c, mask.sum(), score))

pd.DataFrame(out, columns=["country","n","f1"]).sort_values("f1")

,country,n,f1
20,South Korea,11,0.083333
2,Canada,27,0.949343
1,USA,29,1.000000
0,India,36,1.000000
4,Switzerland,23,1.000000
5,Denmark,22,1.000000
6,Turkey,21,1.000000
3,Mexico,25,1.000000
8,Bangladesh,18,1.000000
9,Pakistan,18,1.000000


In [14]:
final_pred = rule(Xte_raw).map({1:"Yes", 0:"No"})
submission = pd.DataFrame({"id": test.id, "addiction": final_pred})

assert len(submission) == 163
assert set(submission.id) == set(sub_sample.id)

submission.to_csv("submission.csv", index=False)
submission.addiction.value_counts()

,count
addiction,
Yes,94
No,69
